# PyTEDI Example Workflow

This notebook provides a simple, self-contained example for running one or more STEDI farm dam scenarios using PyTEDI.

The example assumes the overall project folder is organised as:

```text

├── data/
│   ├── Example_STEDI_scenario.scn
│   ├── Example_STEDI_inflow.fdy
│   └── Example_STEDI_climate.cdy
│   
└── output/
```

The notebook is designed so that inputs remain separate from generated outputs.

## 0. Install the PyTEDI package using pip install
This is only required the first time. It may a few minutes.

In [13]:
#pip install pytedi

## 1. Import required packages

`pytedi` is used to open, run and save the small dam models.

`pandas` is used to manage data inputs and outputs

In [2]:
import pytedi
import pandas as pd

## 2. Import a STEDI model

The key function for reading an existing Windows STEDI scenario is `open_windows_STEDI_scenario()`

This function reads the scenario file, processes the input data, and returns a PyTEDI model object. 

In [3]:
scn_file = r'data\Example_STEDI_scenario.scn'

mymodel = pytedi.open_windows_STEDI_scenario(scn_file)


## 3. Import a STEDI model - advanced options

Many windows STEDI models are defined using a distribution of farm dams. Such a distribution is specified as a histogram, for example with 10% of dams between 0 and 5ML, 15% of dams between 5 and 10ML, and so on. When models are defined in this way, STEDI uses the distribution to randomly generate a list of dams. This is problematic because the random number generator within PyTEDI produces different numbers, and therefore a different list of dams.

The standard function to read a windows STEDI scenario file is `open_windows_STEDI_scenario()`

This will read the scenario file including the information about the dam distribution. PyTEDI will then create a list of dams based on that distribution.

However, if PyTEDI is required to match STEDI more closely, it is possible to read a list of dams directly from a STEDI output file, instead of the distribution from the scenario file. STEDI routinely produces output files listing all dams in the model regardless of how the dams were specified. By reading this output file, PyTEDI can use the exact list of dams in the STEDI model.

To read the list of dams from the STEDI output file, include the option `network_from_out=<outfile>` in the function.


In [4]:
scn_file = r'data\Example_STEDI_scenario.scn'                                              # STEDI scenario file, including the distribution of dams

out_file = r'data\Example_STEDI_out.out'                                                   # STEDI output file including the exact list of dams used in the original calculation, based on the distribution

mymodel = pytedi.open_windows_STEDI_scenario(scn_file, network_from_out=out_file)          # The 'network_from_out' argument reads the exact list of dams from STEDI, instead of the distribution


When randomly generating a list of dams from a distribution, the random number generator within PyTEDI has a fixed seed of 42, which means that the list of dams will be the same on every model run. However, users can vary the seed. This would allow users to generate _different_ lists of dams on each run, in order to quantify uncertainty within the dam distribution and how this affects streamflow.

To vary the seed of the random number generator, use the `seed` option in the function.

In [ ]:
randomseed = 42
mymodel = pytedi.open_windows_STEDI_scenario(scn_file, seed = randomseed)

## 4. Running the model

To run a PyTEDI model, use the `run_model` function.

Note that this command runs the model, but **does not save any outputs**. Further commands are required to save outputs to a file.

In [5]:
mymodel.run_model()

C:\Users\robert.morden\OneDrive - Alluvium Consulting Australia\_Roberts stuff\Tools\Python\pytedi\pytedi_github\pytedi\pytedi\engine.py:104: RuntimeWarning: Reverse solver could not reproduce the observed downstream flow with a non-negative upstream flow on 6 timestep(s). Upstream flow was estimated using recent reconstructed flow, with two-timestep linear extrapolation where available. The largest downstream residual was 22.3034 ML on 1975-10-20.
  global_results, alldam_results = calc_from_down_to_up(damlist_df, demands_df, rain_df, evap_df, flowout_df, cat_area, scale_limit, histoscale_df, store_params)


## 5. Saving results

PyTEDI can produce two types of model results files:

- **Global catchment results** – this includes all major fluxes within the overall catchment, such as inflows, outflows, demands, climate, etc.
- **Individual dam results** – this includes all major fluxes for each individual dam. Results for individual dams are only produced if they have been specifically flagged for output.

To write global catchment model results to a file, use the `save_results_global` function.

To export results for one or more specific dams, use the `save_results_individual` function.


In [6]:
# save global catchment results
mymodel.save_results_global(r'output\Example_pytedi_catch_results.csv')

# save individual dam results
dam_number = 27                                                      # this is the 27th dam in the “damlist”
mymodel.damlist.loc[dam_number,'output'] = 1                         # set the flag to 1 to produce results
dam_number = 28                                                      # this is the 28th dam in the “damlist”
mymodel.damlist.loc[dam_number,'output'] = 1                         # set the flag to 1 to produce results

mymodel.run_model()                                                  # now run the model

mymodel.save_results_individual(r'output\Example_pytedi_indiv_results.csv')         # save the results for the 27th and 28th dams


C:\Users\robert.morden\OneDrive - Alluvium Consulting Australia\_Roberts stuff\Tools\Python\pytedi\pytedi_github\pytedi\pytedi\engine.py:104: RuntimeWarning: Reverse solver could not reproduce the observed downstream flow with a non-negative upstream flow on 6 timestep(s). Upstream flow was estimated using recent reconstructed flow, with two-timestep linear extrapolation where available. The largest downstream residual was 22.3034 ML on 1975-10-20.
  global_results, alldam_results = calc_from_down_to_up(damlist_df, demands_df, rain_df, evap_df, flowout_df, cat_area, scale_limit, histoscale_df, store_params)


## 6. Saving the model

It is important to save the model so that it can be retrieved and re-run later. To save a PyTEDI model, use the `save_model` function.

This will save three model files as follows:

- **Model parameters** *(model_file_prefix + "_params.scn")* – This contains all model parameters such as the model title, catchment area, and the names of the other two files. The file is saved in TOML format and is easily readable in any text editor such as notepad.
- **Dam list** *(model_file_prefix + "_damlist.csv")* – This contains a comma separated list of all dams in the model and their characteristics.
- **Timeseries** *(model_file_prefix + "_timeseries.csv")* – This contains all timeseries data for the model in comma separated format. This includes inflows, outflows, rainfall, evaporation, and demand patterns.


In [7]:
model_file_prefix = (r'output\Example_pytedi_model')
mymodel.save_model(model_file_prefix)


## 7. Open a saved PyTEDI model

To re-open a PyTEDI model saved using the `save_model` function (see Step 6), use the `open_pytedi_scenario` function.

Note that opening the **params** file will automatically open the **damlist** file and the **timeseries** file as well.

In [8]:
parameter_file = (r'output\Example_pytedi_model_params.scn')

mymodel2 = pytedi.open_pytedi_scenario(parameter_file)       # opens the model and assigns it to ‘mymodel2’


Model file opened (requested prefix: output\Example_pytedi_model_params.scn)


## 8. Other key features

PyTEDI has several important features including two methods of historical scaling, variable start volumes, and variable dam surface areas.

### Historical scaling (method 1)

To automatically scale farm dam impacts over time, the user can enter a scaling sequence using the `histoscale` attribute. The `histoscale` dataframe represents changes in the effective level of dam development over time. PyTEDI applies the scaling within the water-balance calculations, including effective capacity, surface area and demand, rather than multiplying the final dam-impact results by a scaling factor.

By default, historical scaling is set to 1.0 (ie. no scaling) from 1 January 1800 to 1 January 2100. To modify the default values, change the `histoscale` attribute.


In [ ]:
histdates = ['1965-01-01', '1990-01-01', '2010-01-01', '2030-01-01']      # identify the key dates for historical development (1965, 1990, 2010, 2030)
histdates = pd.DatetimeIndex(histdates)                                   # convert to a datetimeindex

histvalues = [0.0, 0.4, 1.0, 1.0]                                         # identify levels of development (0% in 1965, 40% in 1990, 100% in 2010 onwards)

mymodel2.histoscale = pd.DataFrame(data=histvalues,index=histdates, columns=['histoscale'], dtype='float64')     # assign the dates and values to histoscale

mymodel2.run_model()                                                      # now run the model


C:\Users\robert.morden\OneDrive - Alluvium Consulting Australia\_Roberts stuff\Tools\Python\pytedi\pytedi_github\pytedi\pytedi\engine.py:104: RuntimeWarning: Reverse solver could not reproduce the observed downstream flow with a non-negative upstream flow on 8 timestep(s). Upstream flow was estimated using recent reconstructed flow, with two-timestep linear extrapolation where available. The largest downstream residual was 13.5676 ML on 1992-11-20.
  global_results, alldam_results = calc_from_down_to_up(damlist_df, demands_df, rain_df, evap_df, flowout_df, cat_area, scale_limit, histoscale_df, store_params)


### Historical scaling (method 2)

The construction and decommissioning dates of each dam can be defined, so that the dam is considered to have zero volume and zero surface area before construction or after decommissioning.

By default, all dams are assumed to be constructed on 1 Jan in the year 0, and decommissioned on 1 Jan in the year 9999.

Note that when a dam is decommissioned, any water held in the dam will be spilled on 1st January.

To set the dates, use the `year_start` and `year_end` fields in the dam list.


In [ ]:
mymodel2.damlist.loc[5,'year_start'] = 1984      # sets the construction date of dam number 5 to 1984

mymodel2.damlist.loc[12,'year_end'] = 2003       # sets the decommission date of dam number 12 to 2003

mymodel2.damlist.loc[:,'year_start'] = 1965      # sets the construction date of all dams to 1965


### Variable start volumes

By default, the Windows STEDI tool assumes all dams begin the model **completely empty**. This is the default in pytedi too.

To vary the start volume, the `startstore_propfull` attribute can be set to any number between 0 and 1.


In [11]:
mymodel2.startstore_propfull = 0.5      # sets all dams to start 50% full

mymodel2.startstore_propfull = 1.0      # sets all dams to start 100% full


### Variable dam surface areas

By default, PyTEDI assumes that the surface area of a dam is the maximum as specified in the dam list. This means that the maximum surface area is assumed for all rainfall and evaporation calculations.

This behaviour is controlled by the `vary_surfarea` attribute which is set to `False` by default. 

To switch on variable surface areas, set `vary_surfarea` attribute to `True`. This will change the calculations so that the surface area of each dam is calculated based on a volume-to-surface area equation given by the parameters a and b as follows:

$$
\text{Volume (ML)} = a \left[\text{Surface area (m}^2\text{)}\right]^b
$$


By default, a = 0.0001042 and b = 1.3213 as given in Fowler et al. (2015).
This example shows how to set the model to vary surface areas and change the equation parameters.


In [12]:
mymodel2.vary_surfarea = True                 # sets surface areas to be calculated dynamically

mymodel2.vol_surfarea_a = 0.000145            # as per Lowe et al. (2005)
mymodel2.vol_surfarea_b = 1.314               # as per Lowe et al. (2005)


## References

Fowler, K., Morden, R., Lowe, L., Nathan, R., 2015. Advances in assessing the impact of hillside farm dams on streamflow. Australian Journal of Water Resources 19. https://doi.org/10.1080/13241583.2015.1116182

Lowe, L., Nathan, R., Morden, R., 2005. Assessing the impact of farm dams on streamflows, Part II: Regional characterisation. Australasian Journal of Water Resources 9, 13–26. https://doi.org/10.1080/13241583.2005.11465260
